## Chaotic scenario 2: Price Divergence Shock

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from amm_basics import *

STEPS = 300
INIT_PRICE = 2000.0
INIT_ETH = 50.0
INIT_DAI = 100_000.0
CRASH_START = 200
ARB_SPEED = 0.40
CRASH_SEVERITY = 0.06
FEE = 0.003


# 1. TOKENS, WALLETS Y POOL

eth = AtomicToken("ETH")
dai = AtomicToken("DAI")

arb_wallet = Wallet("Arbitrageur")
arb_wallet.deposit(eth, 1e9)
arb_wallet.deposit(dai, 1e9)

amm = AMM(eth, dai, UniswapV2(fee=FEE), reserve0=INIT_ETH, reserve1=INIT_DAI)
state = State([arb_wallet], [amm])

# 2. SYNTHETIC MARKET PRICE
rng = np.random.default_rng(42)
mkt_prices = [INIT_PRICE]
for i in range(1, STEPS):
    if i < CRASH_START:
        mkt_prices.append(max(1.0, mkt_prices[-1] + rng.normal(0, 30)))
    else:
        drop = rng.uniform(CRASH_SEVERITY * 0.5, CRASH_SEVERITY)
        mkt_prices.append(max(1.0, mkt_prices[-1] * (1 - drop)))
mkt_prices = np.array(mkt_prices)

# 3. SIMULACIÓN

mkt_hist, pool_hist, il_hist, div_hist, arb_hist = [], [], [], [], []

for i, mp in enumerate(mkt_prices):
    res = amm.reserves
    p_pool = res[dai] / res[eth]

    p_bid = p_pool * (1 - FEE)
    p_ask = p_pool / (1 - FEE)

    arb_active = False

    if mp > p_ask:
        arb_active = True
        eth_target = np.sqrt(res[eth] * res[dai] / mp)
        dy_in = (res[eth] * res[dai] / eth_target) - res[dai]
        if dy_in > 0.001:
            tx = Transaction("swap", arb_wallet, dai, eth, dy_in * ARB_SPEED)
            state.swap(tx)

    elif mp < p_bid:
        arb_active = True
        eth_target = np.sqrt(res[eth] * res[dai] / mp)
        dx_in = eth_target - res[eth]
        if dx_in > 0.001:
            tx = Transaction("swap", arb_wallet, eth, dai, dx_in * ARB_SPEED)
            state.swap(tx)

    res = amm.reserves
    p_pool_after = res[dai] / res[eth]

    theta = p_pool_after / INIT_PRICE
    il_val = (2 * np.sqrt(theta) / (1 + theta) - 1) * 100

    mkt_hist.append(mp)
    pool_hist.append(p_pool_after)
    il_hist.append(il_val)
    div_hist.append((mp - p_pool_after) / p_pool_after * 100)
    arb_hist.append(arb_active)

mkt_hist  = np.array(mkt_hist)
pool_hist = np.array(pool_hist)
il_hist   = np.array(il_hist)
div_hist  = np.array(div_hist)

div_min = float(div_hist.min()) * 1.15
div_max = float(div_hist.max()) * 1.15


inactive_zones = []
in_zone, start = False, 0
for i, active in enumerate(arb_hist):
    if not active and not in_zone:
        in_zone, start = True, i
    elif active and in_zone:
        in_zone = False
        inactive_zones.append((start, i))
if in_zone:
    inactive_zones.append((start, STEPS - 1))


fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        "<b>Market Price vs Pool Price</b>",
        "<b>Impermanent Loss (%)</b>",
        "<b>Price Divergence (%)</b>",
    )
)

steps = np.arange(STEPS)

for (x0, x1) in inactive_zones:
    for row in [1, 2, 3]:
        fig.add_vrect(x0=x0, x1=x1, fillcolor="rgba(255,190,30,0.10)",
                      line_width=0, row=row, col=1)

for row in [1, 2, 3]:
    fig.add_vline(x=CRASH_START, line_dash="dot",
                  line_color="rgba(150,50,200,0.4)", line_width=1.2,
                  row=row, col=1)

fig.add_hrect(y0=-FEE * 100, y1=FEE * 100,
              fillcolor="rgba(100,200,100,0.12)", line_width=0, row=3, col=1)
fig.add_hline(y= FEE * 100, line_dash="dot",
              line_color="rgba(50,150,50,0.5)", line_width=1, row=3, col=1)
fig.add_hline(y=-FEE * 100, line_dash="dot",
              line_color="rgba(50,150,50,0.5)", line_width=1, row=3, col=1)

fig.add_trace(go.Scatter(x=[], y=[], mode="lines",
    line=dict(color="#2c7be5", width=2), name="Market Price"), row=1, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode="lines",
    line=dict(color="#e23636", width=2, dash="dot"), name="Pool Price"), row=1, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode="lines",
    line=dict(color="#e07b39", width=2), fill="tozeroy",
    fillcolor="rgba(224,123,57,0.15)", name="IL %"), row=2, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode="lines",
    line=dict(color="#9b59b6", width=2), fill="tozeroy",
    fillcolor="rgba(155,89,182,0.15)", name="Divergence %"), row=3, col=1)

SKIP = 2
frames = []
for i in range(2, STEPS, SKIP):
    xs = steps[:i]
    frames.append(go.Frame(
        data=[
            go.Scatter(x=xs, y=mkt_hist[:i]),
            go.Scatter(x=xs, y=pool_hist[:i]),
            go.Scatter(x=xs, y=il_hist[:i]),
            go.Scatter(x=xs, y=div_hist[:i]),
        ],
        name=f"f{i}",
        traces=[0, 1, 2, 3],
    ))

fig.frames = frames

fig.update_layout(
    height=750,
    template="plotly_white",
    title=dict(
        text=f"<b>AMM Simulation — UniswapV2 Fee {FEE*100:.1f}% · No-Arb Window ±{FEE*100:.1f}%</b>",
        font=dict(size=14), x=0.5
    ),
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.05, x=0.5, xanchor="center", font=dict(size=11)),
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        x=0.02, y=1.06,
        buttons=[
            dict(label="▶  Play",
                 method="animate",
                 args=[None, {"frame": {"duration": 80, "redraw": True},
                              "fromcurrent": True,
                              "transition": {"duration": 0}}]),
            dict(label="⏸  Pause",
                 method="animate",
                 args=[[None], {"frame": {"duration": 0},
                                "mode": "immediate",
                                "transition": {"duration": 0}}])
        ]
    )],
    xaxis=dict(range=[0, STEPS], showgrid=False),
    xaxis2=dict(range=[0, STEPS], showgrid=False),
    xaxis3=dict(range=[0, STEPS], title="Step", showgrid=False),
    yaxis=dict(range=[0, INIT_PRICE * 1.15], title="DAI/ETH", showgrid=False),
    yaxis2=dict(range=[float(il_hist.min()) * 1.15, 5], title="IL (%)", showgrid=False),
    yaxis3=dict(range=[div_min, div_max], title="Divergence (%)", showgrid=False),
)

fig.show()

![Price Divergence Shock](price_divergence_shock.gif)

During the experiment, when the price evolves with random fluctuations around its initial value, the Impermanent Loss remains limited in absolute magnitude. Nevertheless, at the price divergence shock time, the IL drops dramatically, attaining levels close to −78%. Crucially, the pool price never fully converges to the market price during the crash.

### Theorical background

#### The Mechanics of Impermanent Loss (IL)

In a Constant Product Market Maker (CPMM), liquidity providers (LPs) face a specific opportunity cost known as **Impermanent Loss (IL)**, also referred to as divergence loss. This phenomenon arises because the AMM must continuously rebalance its reserves to match the external market price, effectively selling the outperforming asset and buying the underperforming one.

#### Arbitrageur Capacity Constraints and Price Divergence

A key assumption in the standard IL framework is that arbitrageurs instantly close the gap between the pool price and the external market price. In practice, this breaks down under high volatility or sharp directional moves.

When the market falls faster than arbitrageurs can act, the pool price lags behind. At each step $t$, the arbitrageur closes only a fraction $\lambda \in (0, 1]$ of the gap:

$$
P_{\text{pool}}^{(t+1)} = P_{\text{pool}}^{(t)} + \lambda \cdot \left(P_{\text{market}}^{(t)} - P_{\text{pool}}^{(t)}\right)
$$

When $\lambda < 1$ and the market moves faster than $\lambda$ can compensate, a persistent divergence builds up:

$$
\Delta^{(t)} = \frac{P_{\text{market}}^{(t)} - P_{\text{pool}}^{(t)}}{P_{\text{pool}}^{(t)}} \not\to 0
$$

As a result, the LP's portfolio is repriced at a stale pool price rather than the true market price, amplifying the realized loss beyond the theoretical $I(\theta)$. In a sustained crash, this divergence accumulates without closing, exposing the LP to an **irrecoverable divergence loss** on top of standard IL.

#### The Mechanics of Impermanent Loss (IL)

Mathematically, $I(\theta)$ is defined as the relative difference between the current value of the portfolio inside the pool ($v_C$) and the value of a non-rebalanced portfolio held externally ($v_E$):

$$
I(\theta) = \frac{v_C - v_E}{v_E}
= \frac{2\sqrt{\theta}}{1 + \theta} - 1
$$

where $\theta$ represents the price ratio:

$$
\theta = \frac{P_{\text{final}}}{P_{\text{initial}}}
$$

Since the geometric mean $2\sqrt{\theta}$ is always less than or equal to the arithmetic mean $1 + \theta$, it follows that:

$$
I(\theta) \leq 0
$$

This confirms that LPs experience a loss in value relative to HODLing whenever the price diverges from the entry point. Because the inequality holds for any price ratio $\theta \neq 1$, price movements in either direction produce symmetric adverse effects for the provider.

The loss is termed *impermanent* because it represents an unrealized divergence. If the price reverts to its initial level ($\theta = 1$), equilibrium is restored:

$$
v_C = v_E
$$

and the loss disappears.

Thus, $I(\theta)$ captures the intrinsic rebalancing cost of the CPMM model. When trading fees are introduced, the analysis must incorporate a revenue buffer that may offset this cost. For a liquidity position to be profitable, the accumulated fees must exceed the rebalancing cost $I(\theta)$.

Moreover, the presence of fees creates an **arbitrage-free band**, a small range around the current price where arbitrage is not profitable. Within this band, no rebalancing occurs, temporarily shielding the LP from infinitesimal price fluctuations (as illustrated in the Simulation of a CPMM section of this notebook).


However, for sufficiently large and persistent price declines, the divergence loss grows nonlinearly and can quickly dominate fee income. Since  

$$
I(\theta) = \frac{2\sqrt{\theta}}{1 + \theta} - 1,
$$

the function becomes increasingly negative as $\theta \to 0$. In the extreme case, $I(\theta) \to -1$, meaning the LP loses almost 100% of the value relative to HODLing.


### Conclusion

The theoretical analysis confirms that impermanent loss is an intrinsic and unavoidable cost of providing liquidity in a CPMM. Under the standard assumption of instantaneous arbitrage, IL is fully determined by the price ratio $\theta$ and is symmetric in both directions — any deviation from the entry price, whether upward or downward, reduces the LP's value relative to HODLing.

However, the standard framework understates the true exposure of LPs under realistic market conditions. When arbitrageurs cannot close the price gap instantly — due to capital constraints, latency, or risk limits — the pool price persistently lags the market, repricing the LP's portfolio at a stale rate and generating an irrecoverable divergence loss on top of the theoretical $I(\theta)$. This effect is most severe during sustained directional moves, precisely the conditions under which LPs are already most exposed.

Trading fees provide a partial buffer, but their ability to offset IL depends critically on trading volume and the magnitude of price movements. In low-volatility regimes with sufficient volume, fees can dominate. In crash scenarios, the rebalancing cost grows faster than fee revenue can compensate, and the LP position becomes structurally loss-making.